# google_auth_httplib2 smoke test

Tests `google-auth-httplib2` — the httplib2 transport adapter for google-auth.

Note: `google_auth_httplib2` is a single-file module (no `__path__`, no submodules to walk), so the first cell verifies the top-level import instead of running a `pkgutil.walk_packages` sweep.

In [ ]:
# google_auth_httplib2 is a flat single-file module — no submodules to walk.
import google_auth_httplib2 as _pkg_under_test

assert not hasattr(_pkg_under_test, '__path__'), "expected a flat single-file module"
print(f"OK — {_pkg_under_test.__name__} imported (no submodules)")
public = [n for n in dir(_pkg_under_test) if not n.startswith('_')]
print(f"public surface: {public}")
assert 'AuthorizedHttp' in public and 'Request' in public


In [ ]:
# Example 1: the Request adapter implements google.auth's transport Request contract.
import httplib2
from google.auth.transport import Request as TransportRequest

import google_auth_httplib2

request = google_auth_httplib2.Request(httplib2.Http())
print(f"type: {type(request).__name__}")
print(f"is google.auth transport Request: {isinstance(request, TransportRequest)}")
assert isinstance(request, TransportRequest)
assert callable(request)


In [ ]:
# Example 2: AuthorizedHttp construction + credential wiring.
from google.auth.credentials import AnonymousCredentials

import google_auth_httplib2
import httplib2

creds = AnonymousCredentials()
authed = google_auth_httplib2.AuthorizedHttp(creds)

print(f"type: {type(authed).__name__}")
print(f"credentials wired: {authed.credentials is creds}")
print(f"underlying transport: {type(authed.http).__name__}")
assert authed.credentials is creds
assert isinstance(authed.http, httplib2.Http)


In [ ]:
# Example 3: full request round trip via AuthorizedHttp against a local in-process server.
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from google.auth.credentials import AnonymousCredentials

import google_auth_httplib2

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(b'{"ok": true}')
    def log_message(self, *args):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_address[1]}/data"

authed = google_auth_httplib2.AuthorizedHttp(AnonymousCredentials())
response, content = authed.request(url)
server.shutdown()
print(f"status: {response.status}, content: {content}")
assert response.status == 200
assert b'"ok"' in content


In [ ]:
# Example 4: the Request adapter round trip (what google.auth uses for token endpoints).
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import google_auth_httplib2
import httplib2

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"token-endpoint-ok")
    def log_message(self, *args):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_address[1]}/"

request = google_auth_httplib2.Request(httplib2.Http())
resp = request(url, method="GET")
server.shutdown()
print(f"response.status = {resp.status}")
print(f"response.data = {resp.data}")
assert resp.status == 200
assert resp.data == b"token-endpoint-ok"


In [ ]:
# Example 5: AuthorizedHttp wraps and delegates to the underlying httplib2.Http.
from google.auth.credentials import AnonymousCredentials

import google_auth_httplib2
import httplib2

http_obj = httplib2.Http(timeout=7)
authed = google_auth_httplib2.AuthorizedHttp(AnonymousCredentials(), http=http_obj)

# The explicit transport instance is honored...
assert authed.http is http_obj
# ...and httplib2 attributes delegate through the wrapper.
print(f"delegated timeout: {authed.timeout}")
print(f"follow_redirects: {authed.follow_redirects}")
assert authed.timeout == 7
assert authed.connections is http_obj.connections
authed.close()
print("close() delegated cleanly")
